In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np

INPUT_PATH = "drive/MyDrive/EthicsInAI/elenco_accessi_2010_2021.xlsx"
DATE_FORMAT = "%d.%m.%Y %H:%M:%S"

Caricamento fogli

In [9]:
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
{name: df.shape for name, df in sheets.items()}

{'Accessi': (22469, 2),
 'Anagrafiche': (22469, 8),
 'Acc-Dim': (22469, 21),
 'OBI': (444, 3),
 'Problema Principale': (22591, 4),
 'Ordini': (69846, 4),
 'Sintomi': (22469, 12),
 'Dati clinici': (58994, 6),
 'Ricovero': (750, 2),
 'Triage': (22051, 40),
 'Procedure': (22469, 9),
 'Diz_Procedure': (41, 4)}

Anagrafiche

In [10]:
anagrafiche = (
  sheets["Anagrafiche"][["A01_ID_PERSONA", "A01_SESSO", "A01_DATA_NASCITA"]]
  .drop_duplicates(subset="A01_ID_PERSONA")
  .copy()
)
anagrafiche["A01_DATA_NASCITA"] = pd.to_datetime(
  anagrafiche["A01_DATA_NASCITA"], format=DATE_FORMAT, errors="coerce"
)

print("Persone totali: ", len(anagrafiche))

anagrafiche.head()

Persone totali:  17474


,A01_ID_PERSONA,A01_SESSO,A01_DATA_NASCITA
0,CC10033040,F,2010-06-19
1,CC16005850,M,2015-04-27
3,CC12157603,M,2004-12-25
4,CC16001157,M,2013-03-18
6,CC14048810,M,2014-03-01


Acc-dim

In [11]:
ACC_DIM_COLS = [
  "D01_ID_ACCESSO", "D01_ID_PAZIENTE", "D01_DATAORA_ACCETTAZIONE",
  "D30_DESC_GRAVITA", "D01_DATI_RIFERITI",
  "D01_DIAGNOSI_DIMISSIONE", "D01_TERAPIA_DIMISSIONE", "D31_DESC_CAUSALE",
]
acc_dim = sheets["Acc-Dim"][ACC_DIM_COLS].copy()
acc_dim["D01_DATAORA_ACCETTAZIONE"] = pd.to_datetime(
  acc_dim["D01_DATAORA_ACCETTAZIONE"], format=DATE_FORMAT, errors="coerce"
)

print("Visite totali: ", len(acc_dim))
acc_dim.head()

Visite totali:  22469


,D01_ID_ACCESSO,D01_ID_PAZIENTE,D01_DATAORA_ACCETTAZIONE,D30_DESC_GRAVITA,D01_DATI_RIFERITI,D01_DIAGNOSI_DIMISSIONE,D01_TERAPIA_DIMISSIONE,D31_DESC_CAUSALE
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA


Calcolo età

In [12]:
visite = acc_dim.merge(
  anagrafiche, left_on="D01_ID_PAZIENTE", right_on="A01_ID_PERSONA", how="left"
).drop(columns="A01_ID_PERSONA")

adm, dob = visite["D01_DATAORA_ACCETTAZIONE"], visite["A01_DATA_NASCITA"]
already_had_birthday = (
  (adm.dt.month * 100 + adm.dt.day) >= (dob.dt.month * 100 + dob.dt.day)
)
visite["ETA_ALLA_VISITA"] = (
  adm.dt.year - dob.dt.year - (~already_had_birthday).astype("Int64")
).astype("Int64")

visite["ETA_IN_GIORNI_ALLA_VISITA"] = (
  visite["D01_DATAORA_ACCETTAZIONE"] - visite["A01_DATA_NASCITA"]
).dt.days.astype("Int64")

visite.head()

,D01_ID_ACCESSO,D01_ID_PAZIENTE,D01_DATAORA_ACCETTAZIONE,D30_DESC_GRAVITA,D01_DATI_RIFERITI,D01_DIAGNOSI_DIMISSIONE,D01_TERAPIA_DIMISSIONE,D31_DESC_CAUSALE,A01_SESSO,A01_DATA_NASCITA,ETA_ALLA_VISITA,ETA_IN_GIORNI_ALLA_VISITA
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO,M,2006-11-26,3,1132
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO,F,2008-06-06,1,574
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA,M,2006-04-17,3,1355
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA,F,2005-07-18,4,1628
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA,F,2000-02-29,9,3594


OBI

In [13]:
obi_ids = set(sheets["OBI"]["D08_ID_ACCESSO"])
visite["OBI"] = visite["D01_ID_ACCESSO"].isin(obi_ids)
print(f"OBI: ", visite['OBI'].sum())

OBI:  444


Ricovero

In [14]:
ricovero_ids = set(sheets["Ricovero"]["D36_ID_ACCESSO"])
visite["RICOVERO"] = visite["D01_ID_ACCESSO"].isin(ricovero_ids)
print(f"RICOVERO: ", visite['RICOVERO'].sum())

RICOVERO:  750


Problema principale

In [15]:
pp = sheets["Problema Principale"]
pp_agg = (
  pp.dropna(subset=["E35_DESCRIZIONE"])
  .groupby("E36_ID_ACCESSO")["E35_DESCRIZIONE"]
  .agg(" | ".join)
  .reset_index()
  .rename(columns={"E35_DESCRIZIONE": "PROBLEMA_PRINCIPALE"})
)
visite = visite.merge(
  pp_agg, left_on="D01_ID_ACCESSO", right_on="E36_ID_ACCESSO", how="left"
).drop(columns="E36_ID_ACCESSO")

visite['PROBLEMA_PRINCIPALE'] = visite["PROBLEMA_PRINCIPALE"].fillna("NON SPECIFICATO")

visite.head()

,D01_ID_ACCESSO,D01_ID_PAZIENTE,D01_DATAORA_ACCETTAZIONE,D30_DESC_GRAVITA,D01_DATI_RIFERITI,D01_DIAGNOSI_DIMISSIONE,D01_TERAPIA_DIMISSIONE,D31_DESC_CAUSALE,A01_SESSO,A01_DATA_NASCITA,ETA_ALLA_VISITA,ETA_IN_GIORNI_ALLA_VISITA,OBI,RICOVERO,PROBLEMA_PRINCIPALE
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO,M,2006-11-26,3,1132,False,False,NON SPECIFICATO
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO,F,2008-06-06,1,574,False,False,NON SPECIFICATO
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA,M,2006-04-17,3,1355,False,False,NON SPECIFICATO
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA,F,2005-07-18,4,1628,False,False,NON SPECIFICATO
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA,F,2000-02-29,9,3594,False,False,NON SPECIFICATO


Triage

In [16]:
TRIAGE_COLS = [
  "D07_ID_ACCESSO",
  "D07_FREQUENZA_CARDIACA",
  "D07_TEMPERATURA", "D07_SATURAZIONE_OSSIGENO",
  "D07_GCS_PEDIATRICO", "D07_GCS_VALORE",
  "D07_GCS_NEONATALE", "D07_PESO_CORPOREO",
]
triage = sheets["Triage"][TRIAGE_COLS].drop_duplicates(subset="D07_ID_ACCESSO", keep="first")
visite = visite.merge(
  triage, left_on="D01_ID_ACCESSO", right_on="D07_ID_ACCESSO", how="left"
).drop(columns="D07_ID_ACCESSO")

visite.head()

,D01_ID_ACCESSO,D01_ID_PAZIENTE,D01_DATAORA_ACCETTAZIONE,D30_DESC_GRAVITA,D01_DATI_RIFERITI,D01_DIAGNOSI_DIMISSIONE,D01_TERAPIA_DIMISSIONE,D31_DESC_CAUSALE,A01_SESSO,A01_DATA_NASCITA,...,OBI,RICOVERO,PROBLEMA_PRINCIPALE,D07_FREQUENZA_CARDIACA,D07_TEMPERATURA,D07_SATURAZIONE_OSSIGENO,D07_GCS_PEDIATRICO,D07_GCS_VALORE,D07_GCS_NEONATALE,D07_PESO_CORPOREO
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO,M,2006-11-26,...,False,False,NON SPECIFICATO,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO,F,2008-06-06,...,False,False,NON SPECIFICATO,160.0,36.0,100.0,NO,NaN,NO,12.7
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA,M,2006-04-17,...,False,False,NON SPECIFICATO,NaN,36.2,NaN,NO,NaN,NO,NaN
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA,F,2005-07-18,...,False,False,NON SPECIFICATO,NaN,37.4,NaN,NO,NaN,NO,15.0
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA,F,2000-02-29,...,False,False,NON SPECIFICATO,128.0,36.7,99.0,NO,NaN,NO,NaN


Dati clinici

In [17]:
dc = sheets["Dati clinici"][["O03_ID_ACCESSO", "O16_DESC_TIPO_DATO", "O03_TESTO"]].copy()

# Concatenate within (visit, type) duplicates with a single space.
dc_agg = (
  dc.dropna(subset=["O03_TESTO"])
  .assign(O03_TESTO=lambda d: d["O03_TESTO"].astype(str))
  .groupby(["O03_ID_ACCESSO", "O16_DESC_TIPO_DATO"])["O03_TESTO"]
  .agg(" ".join)
  .reset_index()
)

# Pivot type column to wide format.
dc_wide = dc_agg.pivot(
  index="O03_ID_ACCESSO", columns="O16_DESC_TIPO_DATO", values="O03_TESTO"
).reset_index()
dc_wide.columns.name = None

# Standardize column names.
DC_RENAME = {
  "Anamnesi": "ANAMNESI",
  "Esame obiettivo": "NOTE_AGGIUNTIVE",
  "Referto": "REFERTO",
  "Diario clinico": "DIARIO_CLINICO",
}
dc_wide = dc_wide.rename(columns=DC_RENAME)

visite = visite.merge(
  dc_wide, left_on="D01_ID_ACCESSO", right_on="O03_ID_ACCESSO", how="left"
).drop(columns=["O03_ID_ACCESSO", "REFERTO", "DIARIO_CLINICO"])



visite.head()

,D01_ID_ACCESSO,D01_ID_PAZIENTE,D01_DATAORA_ACCETTAZIONE,D30_DESC_GRAVITA,D01_DATI_RIFERITI,D01_DIAGNOSI_DIMISSIONE,D01_TERAPIA_DIMISSIONE,D31_DESC_CAUSALE,A01_SESSO,A01_DATA_NASCITA,...,PROBLEMA_PRINCIPALE,D07_FREQUENZA_CARDIACA,D07_TEMPERATURA,D07_SATURAZIONE_OSSIGENO,D07_GCS_PEDIATRICO,D07_GCS_VALORE,D07_GCS_NEONATALE,D07_PESO_CORPOREO,ANAMNESI,NOTE_AGGIUNTIVE
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO,M,2006-11-26,...,NON SPECIFICATO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I genitori riferiscono che mentre giocava con ...,Presenza di ustione di I grado sulla faccia vo...
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO,F,2008-06-06,...,NON SPECIFICATO,160.0,36.0,100.0,NO,NaN,NO,12.7,Giunge in PS per lesione orticariodi comparse ...,Buone condizioni generali. Vigile e reattivo. ...
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA,M,2006-04-17,...,NON SPECIFICATO,NaN,36.2,NaN,NO,NaN,NO,NaN,Da ieri sera comparsa di lesioni pruriginose m...,Presenza di lesioni pruriginose maculo-papulos...
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA,F,2005-07-18,...,NON SPECIFICATO,NaN,37.4,NaN,NO,NaN,NO,15.0,Da ieri comparsa di lesioni vescicolose e macu...,Multiple lesioni vescicolose e maculo-papulose...
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA,F,2000-02-29,...,NON SPECIFICATO,128.0,36.7,99.0,NO,NaN,NO,NaN,Giunge in PS per la comparsa di bruciore a liv...,Discrete condizioni generali. Disturbata da pr...


Sanity check

In [18]:
print(f"Shape: {visite.shape}")
print(f"Unique visit IDs: {visite['D01_ID_ACCESSO'].nunique():,}")
print(f"Unique patients : {visite['D01_ID_PAZIENTE'].nunique():,}")

print("\nMissing values per column:")
print(visite.isna().sum().sort_values(ascending=False).to_string())

print("\nAge anomalies:")
adult = visite["ETA_ALLA_VISITA"] >= 18
suspect_dob = visite["A01_DATA_NASCITA"].dt.year < 1990
print(f"  age >= 18: {adult.sum():>4}")
print(f"  age >= 30: {(visite['ETA_ALLA_VISITA'] >= 30).sum():>4}")
print(f"  DOB year < 1990: {suspect_dob.sum():>4}")
print(f"  missing AGE: {visite['ETA_ALLA_VISITA'].isna().sum():>4}")

Shape: (22469, 24)
Unique visit IDs: 22,469
Unique patients : 17,474

Missing values per column:
D31_DESC_CAUSALE             14219
D07_FREQUENZA_CARDIACA        3495
D07_PESO_CORPOREO             3372
D07_SATURAZIONE_OSSIGENO      3028
D07_GCS_VALORE                2206
D07_TEMPERATURA               1099
NOTE_AGGIUNTIVE               1099
D01_TERAPIA_DIMISSIONE        1050
ANAMNESI                       776
D01_DIAGNOSI_DIMISSIONE        682
D07_GCS_NEONATALE              418
D07_GCS_PEDIATRICO             418
D01_ID_ACCESSO                   0
D01_ID_PAZIENTE                  0
D01_DATAORA_ACCETTAZIONE         0
D30_DESC_GRAVITA                 0
D01_DATI_RIFERITI                0
PROBLEMA_PRINCIPALE              0
OBI                              0
RICOVERO                         0
A01_DATA_NASCITA                 0
ETA_ALLA_VISITA                  0
A01_SESSO                        0
ETA_IN_GIORNI_ALLA_VISITA        0

Age anomalies:
  age >= 18:   35
  age >= 30:   15
  DOB year 

Outlier e NA filling

In [19]:
# Età
n_before = len(visite)
visite = visite[visite["ETA_ALLA_VISITA"] < 18].reset_index(drop=True)
print(f"Rimosse {n_before - len(visite)}")

Rimosse 35


In [21]:
# Temperatura
high = visite["D07_TEMPERATURA"] > 50
visite.loc[high, "D07_TEMPERATURA"] /= 10
n_rescaled = high.sum()

low = visite["D07_TEMPERATURA"] < 30
n_temp_nan = low.sum()
visite.loc[low, "D07_TEMPERATURA"] = np.nan

# Saturazione ossigeno
sp_low = visite["D07_SATURAZIONE_OSSIGENO"] < 70
n_spo2_nan = sp_low.sum()
visite.loc[sp_low, "D07_SATURAZIONE_OSSIGENO"] = np.nan

print(f"Temperatura: {n_rescaled} riscalate, {n_temp_nan} -> NaN")
print(f"Sat.: {n_spo2_nan} -> NaN")
print(f"\nTemperatura post-fix:  min={visite['D07_TEMPERATURA'].min():.1f}, "
  f"max={visite['D07_TEMPERATURA'].max():.1f}, "
  f"median={visite['D07_TEMPERATURA'].median():.1f}")
print(f"Sat. post-fix:  min={visite['D07_SATURAZIONE_OSSIGENO'].min():.0f}, "
  f"max={visite['D07_SATURAZIONE_OSSIGENO'].max():.0f}, "
  f"median={visite['D07_SATURAZIONE_OSSIGENO'].median():.0f}")

Temperatura: 0 riscalate, 0 -> NaN
Sat.: 0 -> NaN

Temperatura post-fix:  min=30.0, max=40.9, median=36.2
Sat. post-fix:  min=75, max=100, median=99


In [22]:
for col in ["D07_FREQUENZA_CARDIACA", "D07_PESO_CORPOREO"]:
  by_age = visite.groupby("ETA_ALLA_VISITA")[col].transform("median")
  overall = visite[col].median()
  visite[col] = visite[col].fillna(by_age).fillna(overall)

# Constant clinical defaults for age-independent vitals
visite["D07_TEMPERATURA"] = visite["D07_TEMPERATURA"].fillna(36.5)
visite["D07_SATURAZIONE_OSSIGENO"] = visite["D07_SATURAZIONE_OSSIGENO"].fillna(99.0)
visite["D07_GCS_VALORE"] = visite["D07_GCS_VALORE"].fillna(15.0)

num_cols = ["D07_FREQUENZA_CARDIACA", "D07_PESO_CORPOREO", "D07_TEMPERATURA",
            "D07_SATURAZIONE_OSSIGENO", "D07_GCS_VALORE"]
print("Conto NA rimaste:")
print(visite[num_cols].isna().sum().to_string())

Conto NA rimaste:
D07_FREQUENZA_CARDIACA      0
D07_PESO_CORPOREO           0
D07_TEMPERATURA             0
D07_SATURAZIONE_OSSIGENO    0
D07_GCS_VALORE              0


In [23]:
visite["D07_GCS_PEDIATRICO"] = visite["D07_GCS_PEDIATRICO"].fillna("NO")
visite["D07_GCS_NEONATALE"]  = visite["D07_GCS_NEONATALE"].fillna("NO")
visite["D31_DESC_CAUSALE"] = visite["D31_DESC_CAUSALE"].fillna("NON PRESENTE")

text_cols = ["ANAMNESI", "NOTE_AGGIUNTIVE",
             "D01_DIAGNOSI_DIMISSIONE", "D01_TERAPIA_DIMISSIONE"]
for col in text_cols:
  visite[col] = visite[col].fillna("")

print("Sanity check NA:")
print(visite.isna().sum().sort_values(ascending=False).to_string())
print(f"\nFinal shape: {visite.shape}")

Sanity check NA:
D01_ID_ACCESSO               0
D01_ID_PAZIENTE              0
D01_DATAORA_ACCETTAZIONE     0
D30_DESC_GRAVITA             0
D01_DATI_RIFERITI            0
D01_DIAGNOSI_DIMISSIONE      0
D01_TERAPIA_DIMISSIONE       0
D31_DESC_CAUSALE             0
A01_SESSO                    0
A01_DATA_NASCITA             0
ETA_ALLA_VISITA              0
ETA_IN_GIORNI_ALLA_VISITA    0
OBI                          0
RICOVERO                     0
PROBLEMA_PRINCIPALE          0
D07_FREQUENZA_CARDIACA       0
D07_TEMPERATURA              0
D07_SATURAZIONE_OSSIGENO     0
D07_GCS_PEDIATRICO           0
D07_GCS_VALORE               0
D07_GCS_NEONATALE            0
D07_PESO_CORPOREO            0
ANAMNESI                     0
NOTE_AGGIUNTIVE              0

Final shape: (22434, 24)


In [24]:
visite.rename(columns={'D01_ID_ACCESSO': 'ID_ACCESSO',
                       'D01_ID_PAZIENTE': 'ID_PAZIENTE',
                       'D01_DATAORA_ACCETTAZIONE': 'DATA_ACCETTAZIONE',
                       'D30_DESC_GRAVITA': 'GRAVITA',
                       'D01_DATI_RIFERITI': 'DATI_RIFERITI',
                       'D01_DIAGNOSI_DIMISSIONE': 'DIAGNOSI',
                       'D31_DESC_CAUSALE': 'CAUSALE',
                       'D01_TERAPIA_DIMISSIONE': 'TERAPIA',
                       'A01_SESSO': 'SESSO',
                       'A01_DATA_NASCITA': 'DATA_NASCITA',
                       'D07_FREQUENZA_CARDIACA': 'FREQUENZA_CARDIACA',
                       'D07_TEMPERATURA': 'TEMPERATURA',
                       'D07_SATURAZIONE_OSSIGENO': 'SATURAZIONE_OSSIGENO',
                       'D07_GCS_PEDIATRICO': 'GCS_PEDIATRICO',
                       'D07_GCS_VALORE': 'GCS_VALORE',
                       'D07_GCS_NEONATALE': 'GCS_NEONATALE',
                       'D07_PESO_CORPOREO': 'PESO_CORPOREO',}, inplace=True)

In [25]:
visite.describe()

,DATA_ACCETTAZIONE,DATA_NASCITA,ETA_ALLA_VISITA,ETA_IN_GIORNI_ALLA_VISITA,FREQUENZA_CARDIACA,TEMPERATURA,SATURAZIONE_OSSIGENO,GCS_VALORE,PESO_CORPOREO
count,22434,22434,22434.0,22434.0,22434.000000,22434.000000,22434.000000,22434.000000,22434.000000
mean,2015-08-25 14:28:10.230097408,2011-05-12 08:21:41.738432768,3.798386,1565.644557,116.001582,36.463974,98.795712,14.950611,28.095497
min,2010-01-01 01:09:45,1993-01-15 00:00:00,0.0,1.0,2.000000,30.000000,75.000000,7.000000,1.400000
25%,2012-12-31 11:36:58.249999872,2008-06-25 00:00:00,1.0,498.0,101.000000,36.000000,98.000000,15.000000,11.000000
50%,2015-07-30 19:04:46,2011-08-06 00:00:00,3.0,1145.0,115.000000,36.200000,99.000000,15.000000,15.000000
75%,2018-05-06 10:07:49.249999872,2014-11-17 00:00:00,6.0,2331.0,128.000000,36.700000,100.000000,15.000000,23.000000
max,2021-12-31 22:48:46,2021-11-17 00:00:00,17.0,6396.0,250.000000,40.900000,100.000000,15.000000,10250.000000
std,NaN,NaN,3.635226,1329.994191,20.344728,0.651808,1.068733,0.260240,236.249491


In [26]:
visite.head()

,ID_ACCESSO,ID_PAZIENTE,DATA_ACCETTAZIONE,GRAVITA,DATI_RIFERITI,DIAGNOSI,TERAPIA,CAUSALE,SESSO,DATA_NASCITA,...,PROBLEMA_PRINCIPALE,FREQUENZA_CARDIACA,TEMPERATURA,SATURAZIONE_OSSIGENO,GCS_PEDIATRICO,GCS_VALORE,GCS_NEONATALE,PESO_CORPOREO,ANAMNESI,NOTE_AGGIUNTIVE
0,PS10000009,CC06086820,2010-01-01 01:09:45,BIANCO,lieve ustione mano sx,Ustione di I grado 3° e 4° dito mano sin,Controllo presso Ambulatorio Pediatrico di Der...,ALTRO,M,2006-11-26,...,NON SPECIFICATO,115.0,36.5,99.0,NO,15.0,NO,15.0,I genitori riferiscono che mentre giocava con ...,Presenza di ustione di I grado sulla faccia vo...
1,PS10000056,CC10000036,2010-01-01 05:52:04,VERDE,eruzione cutanea,orticaria acuta,"SI consiglia - bentelan cp 0,5 mg: 1 cp alla m...",ALTRO,F,2008-06-06,...,NON SPECIFICATO,160.0,36.0,100.0,NO,15.0,NO,12.7,Giunge in PS per lesione orticariodi comparse ...,Buone condizioni generali. Vigile e reattivo. ...
2,PS10000144,CC06052311,2010-01-01 12:13:02,BIANCO,Sospetta varicella,Varicella,Fenisitil 10 gocce per 3 volte/die fino a riso...,MALATTIA,M,2006-04-17,...,NON SPECIFICATO,115.0,36.2,99.0,NO,15.0,NO,15.0,Da ieri sera comparsa di lesioni pruriginose m...,Presenza di lesioni pruriginose maculo-papulos...
3,PS10000153,CC09012862,2010-01-01 12:36:02,BIANCO,Sospetta varicella,Varicella,- Fenistil gocce: 15 gocce x 3 volte al giorno...,MALATTIA,F,2005-07-18,...,NON SPECIFICATO,110.0,37.4,99.0,NO,15.0,NO,15.0,Da ieri comparsa di lesioni vescicolose e macu...,Multiple lesioni vescicolose e maculo-papulose...
4,PS10000202,CC05037829,2010-01-01 14:34:26,VERDE,sospetta allergia alla nociolina presenta orti...,Reazione allergica ad alimenti,"- Deltacortene cp 25 mg: 1 cp questa sera, 1/2...",MALATTIA,F,2000-02-29,...,NON SPECIFICATO,128.0,36.7,99.0,NO,15.0,NO,33.0,Giunge in PS per la comparsa di bruciore a liv...,Discrete condizioni generali. Disturbata da pr...


Identificazione NAP

In [27]:
# Tier A: terminologia esplicita di maltrattamento/abuso
TIER_A_PATTERNS = [
    r"\bmaltrattament\w*",
    r"\babuso\s+(sessuale|fisico|minorile|sui\s+minori|su\s+minor)",
    r"\bviolenza\s+(sessuale|fisica|domestic\w+)",
    r"\bpercoss\w+",
    r"\btrauma\s+non\s+accidental\w*",
    r"\bpatologia\s+non\s+accidental\w*",
    r"\blesion\w+\s+non\s+accidental\w*",
    r"\bsindrome\s+del\s+bambino\s+scosso",
    r"\bshaken\s+baby", r"\bmunchausen", r"\bbattered\s+child",
]

# NAP standalone: cattura "NAP" come Tier A
NAP_CONFIRM = r"\bnap\b"
NAP_NEGATE  = (
    r"\b(sospett\w+|possibil\w+|esclus\w+|escludere|rule\s*out|screening\s+(per\s+)?)"
    r"(\s+di\s+|\s+la\s+|\s+)?nap\b"
    r"|\bnap\s+(sospett|esclus|negativ)"
)

# Tier B: linguaggio tentativo o incongruenze
TIER_B_PATTERNS = [
    r"\bsospett\w+\s+(di\s+)?(abus|maltratt|violenz|percoss|trauma\s+non\s+accidental|nap\b)",
    r"\b(abus|maltratt|violenz)\w*\s+sospett",
    r"\bscreening\s+(per\s+)?nap\b",
    r"\bnon\s+(compatibile|congruente|coerente|plausibile)\s+con\s+(la\s+|l['’]\s*)?(anamnesi|meccanismo|dinamica|riferito)",
    r"\bincompatibil\w+\s+con\s+(la\s+|l['’]\s*)?(anamnesi|meccanismo|dinamica)",
    r"\bincongru\w+\s+con\s+(la\s+|l['’]\s*)?(anamnesi|meccanismo|dinamica|riferito)",
    r"\banamnesi\s+(non\s+chiara|confusa|contraddittoria|incongruente)",
    r"\bdinamica\s+(non\s+chiara|poco\s+chiara|non\s+plausibile)",
]

# Tier C: segnali sociali/legali isolati
TIER_C_PATTERNS = [
    r"\bservizi\s+social\w+",
    r"\btribunale\s+(per\s+i\s+minoren\w+|dei\s+minor\w*)",
    r"\bprocura\s+(della\s+repubblica\s+)?(per\s+i\s+minoren\w+|dei\s+minor\w*|minor\w*)",
    r"\bsegnala\w*\s+(ai|al|alla)\s+(servizi|autorit|procur|tribunal)",
    r"\btutela\s+(del|della)?\s*minor",
    r"\bdenuncia",
    r"\bcodice\s+rosa",
]

In [29]:
TEXT_COLS = ["DIAGNOSI", "TERAPIA", "DATI_RIFERITI", "ANAMNESI", "NOTE_AGGIUNTIVE"]
text_low = {c: visite[c].fillna("").astype(str).str.lower() for c in TEXT_COLS}

def any_match(cols, patterns):
    """True se almeno uno dei pattern matcha almeno una delle colonne, per ogni riga."""
    regex = "|".join(patterns)
    masks = [text_low[c].str.contains(regex, regex=True, na=False) for c in cols]
    return np.logical_or.reduce(masks)

# Tier A: pattern espliciti in DIAGNOSI/TERAPIA, NAP standalone (non negato), causale AGGRESSIONE
explicit_a    = any_match(["DIAGNOSI", "TERAPIA"], TIER_A_PATTERNS)
nap_any       = any_match(TEXT_COLS, [NAP_CONFIRM])
nap_negated   = any_match(TEXT_COLS, [NAP_NEGATE])
nap_confirmed = nap_any & ~nap_negated
caus_aggr     = visite["CAUSALE"].fillna("").str.contains("AGGRESSIONE", regex=False)
tier_a = explicit_a | nap_confirmed | caus_aggr

# Tier B e C: esclusivi rispetto ai tier precedenti
tier_b = any_match(TEXT_COLS, TIER_B_PATTERNS) & ~tier_a
tier_c = any_match(TEXT_COLS, TIER_C_PATTERNS) & ~tier_a & ~tier_b

# Etichetta finale
visite["NAP_LABEL"] = "negativo"
visite.loc[tier_c, "NAP_LABEL"] = "segnale_sociale"
visite.loc[tier_b, "NAP_LABEL"] = "sospetto"
visite.loc[tier_a, "NAP_LABEL"] = "confermato"

print("Distribuzione etichette:")
print(visite["NAP_LABEL"].value_counts().to_string())

/tmp/ipykernel_2390/768280846.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  masks = [text_low[c].str.contains(regex, regex=True, na=False) for c in cols]


Distribuzione etichette:
NAP_LABEL
negativo           22351
confermato            60
segnale_sociale       15
sospetto               8


In [32]:
OUTPUT_DIR = "drive/MyDrive/EthicsInAI"

for col in visite.select_dtypes(include="object").columns:
    visite[col] = visite[col].where(visite[col].isna(), visite[col].astype(str))

sicuri   = visite[visite["NAP_LABEL"] == "confermato"].copy()
sospetti = visite[visite["NAP_LABEL"].isin(["sospetto", "segnale_sociale"])].copy()
negativi = visite[visite["NAP_LABEL"] == "negativo"].copy()

sicuri.to_parquet("nap_sicuri.parquet", index=False)
sospetti.to_parquet("nap_sospetti.parquet", index=False)
negativi.to_parquet("nap_negativi.parquet", index=False)

print(f"Sicuri:   {len(sicuri):>6,} visite  →  nap_sicuri.parquet")
print(f"Sospetti: {len(sospetti):>6,} visite  →  nap_sospetti.parquet")
print(f"Negativi: {len(negativi):>6,} visite  →  nap_negativi.parquet")

Sicuri:       60 visite  →  nap_sicuri.parquet
Sospetti:     23 visite  →  nap_sospetti.parquet
Negativi: 22,351 visite  →  nap_negativi.parquet
